# System Handler — Reel 13 Demo  
## ¿Qué es una Vector Database?

Este notebook está diseñado para explicar, de forma visual y práctica, la idea central del reel 13 de la semana 3:

> **Una base de datos tradicional busca palabras exactas.  
> Una vector database busca significado.**

### Qué vas a ver
1. Por qué falla una búsqueda exacta  
2. Cómo convertimos texto en embeddings  
3. Cómo visualizamos vectores  
4. Cómo guardar embeddings en una vector database  
5. Cómo recuperar resultados por similitud semántica  

### Librerías usadas
Este notebook está alineado con el `pyproject.toml` del proyecto y usa un stack compatible con:
- `openai`
- `chromadb`
- `plotly`
- `scikit-learn`
- `numpy`
- `pandas`

### Requisitos
- Tener configurada la variable de entorno `OPENAI_API_KEY`
- Tener instaladas las dependencias del proyecto

> Si no tienes API key, igual puedes leer la lógica del notebook.  
> Las celdas de embeddings y búsqueda semántica requieren acceso al modelo de embeddings.


## 1) Imports y validación básica

Aquí cargamos únicamente librerías que ya existen en el proyecto.  
La idea es evitar incompatibilidades y mantener el demo simple, claro y reproducible.


In [ ]:
import os
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from openai import OpenAI
import chromadb

print("Imports cargados correctamente.")

## 2) Dataset pequeño y entendible

Vamos a usar textos cortos, fáciles de comparar.

La intención pedagógica es mostrar tres cosas:
- textos con palabras distintas pero significado parecido,
- textos con palabras parecidas,
- y textos totalmente no relacionados.


In [ ]:
documents = [
    {"id": "doc_1", "text": "Ventas del primer trimestre", "category": "finanzas", "language": "es"},
    {"id": "doc_2", "text": "Ingresos del Q1", "category": "finanzas", "language": "es"},
    {"id": "doc_3", "text": "Política de reembolso para clientes", "category": "soporte", "language": "es"},
    {"id": "doc_4", "text": "Condiciones de devolución de productos", "category": "soporte", "language": "es"},
    {"id": "doc_5", "text": "Receta de pasta italiana casera", "category": "cocina", "language": "es"},
    {"id": "doc_6", "text": "Manual de seguridad industrial", "category": "seguridad", "language": "es"},
    {"id": "doc_7", "text": "Revenue report for the first quarter", "category": "finanzas", "language": "en"},
    {"id": "doc_8", "text": "Customer refund policy", "category": "soporte", "language": "en"},
]

df_docs = pd.DataFrame(documents)
df_docs

## 3) Simulación de búsqueda tradicional

Una búsqueda tradicional depende de coincidencias directas de texto.

Eso significa que si preguntas algo con palabras diferentes, aunque el significado sea muy parecido, la búsqueda puede fallar.


In [ ]:
def keyword_search(query: str, docs: List[Dict[str, Any]]) -> pd.DataFrame:
    query_tokens = set(query.lower().split())
    rows = []
    for doc in docs:
        doc_tokens = set(doc["text"].lower().split())
        overlap = len(query_tokens.intersection(doc_tokens))
        rows.append({
            "id": doc["id"],
            "text": doc["text"],
            "keyword_overlap": overlap
        })
    return pd.DataFrame(rows).sort_values(by=["keyword_overlap", "id"], ascending=[False, True]).reset_index(drop=True)

query_keyword = "ingresos trimestrales"
keyword_results = keyword_search(query_keyword, documents)
keyword_results

### Observación

Si una base normal depende solo de palabras exactas, puede no encontrar bien algo como:

- **"Ventas del primer trimestre"**
- **"Ingresos del Q1"**

aunque para una persona ambos textos estén muy relacionados.

Ahora vamos a cambiar de enfoque:
en lugar de buscar palabras, vamos a buscar **significado**.


## 4) Cliente OpenAI y función para embeddings

Usamos la sintaxis compatible con `openai>=1.x`, que es la que aparece en tu proyecto.

Este notebook usa el modelo:

- `text-embedding-3-small`

Si la variable `OPENAI_API_KEY` no existe, esta celda te avisará antes de continuar.


In [ ]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise EnvironmentError(
        "No se encontró OPENAI_API_KEY. Configúrala antes de ejecutar las celdas de embeddings."
    )

client = OpenAI(api_key=api_key)

def get_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> List[List[float]]:
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [item.embedding for item in response.data]

print("Cliente OpenAI listo.")

## 5) Generar embeddings

Aquí ocurre la transformación clave:

**texto → vector numérico**

Ese vector representa el significado del texto en un espacio matemático de muchas dimensiones.


In [ ]:
texts = [doc["text"] for doc in documents]
embeddings = get_embeddings(texts)

len(embeddings), len(embeddings[0])

## 6) Inspeccionar un embedding

No vamos a imprimir el vector completo porque puede ser largo.
Solo veremos:
- el tipo de dato,
- su longitud,
- y los primeros valores.

Esto ayuda a aterrizar la idea de que el texto ahora vive como números.


In [ ]:
first_embedding = embeddings[0]

print("Tipo:", type(first_embedding))
print("Dimensiones:", len(first_embedding))
print("Primeros 12 valores:")
first_embedding[:12]

## 7) Reducir dimensiones para visualizar

Los embeddings reales viven en muchas dimensiones.

Como eso no se puede dibujar directamente, usamos **PCA** para proyectarlos a 2 dimensiones y poder verlos en una gráfica.

> Importante: esta visualización es una simplificación pedagógica, no el espacio real completo.


In [ ]:
embedding_matrix = np.array(embeddings)

pca = PCA(n_components=2)
points_2d = pca.fit_transform(embedding_matrix)

df_plot = df_docs.copy()
df_plot["x"] = points_2d[:, 0]
df_plot["y"] = points_2d[:, 1]
df_plot

## 8) Visualización interactiva de embeddings

Aquí deberías notar que textos relacionados tienden a quedar más cerca en el mapa.


In [ ]:
fig = go.Figure()

for category in df_plot["category"].unique():
    subset = df_plot[df_plot["category"] == category]
    fig.add_trace(
        go.Scatter(
            x=subset["x"],
            y=subset["y"],
            mode="markers+text",
            text=subset["text"],
            textposition="top center",
            name=category,
            customdata=subset[["id", "language"]],
            hovertemplate="<b>%{text}</b><br>ID: %{customdata[0]}<br>Idioma: %{customdata[1]}<extra></extra>"
        )
    )

fig.update_layout(
    title="Embeddings proyectados en 2D",
    xaxis_title="Componente principal 1",
    yaxis_title="Componente principal 2",
    height=700
)

fig.show()

## 9) Crear una Vector Database con ChromaDB

Ahora sí vamos a guardar nuestros documentos junto con sus embeddings en una colección vectorial.

Usaremos un cliente en memoria para que el demo sea simple y no dependa de archivos externos.


In [ ]:
chroma_client = chromadb.Client()
collection_name = "system_handler_vector_demo"

existing = [c.name for c in chroma_client.list_collections()]
if collection_name in existing:
    chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)

collection.add(
    ids=[doc["id"] for doc in documents],
    documents=[doc["text"] for doc in documents],
    embeddings=embeddings,
    metadatas=[
        {"category": doc["category"], "language": doc["language"]}
        for doc in documents
    ]
)

print("Colección creada:", collection_name)
print("Total de documentos:", collection.count())

## 10) Consulta semántica

Ahora vamos a hacer una pregunta usando palabras diferentes a las del dataset original.

La idea es comprobar si la vector database puede recuperar significado, no solo coincidencia exacta.


In [ ]:
query_semantic = "¿Cuáles fueron los ingresos del trimestre?"
query_embedding = get_embeddings([query_semantic])[0]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "distances", "metadatas"]
)

results

## 11) Presentar resultados de forma amigable

Vamos a convertir la respuesta del vector store en una tabla más fácil de leer.


In [ ]:
semantic_rows = []
for rank, (doc_text, distance, metadata) in enumerate(
    zip(results["documents"][0], results["distances"][0], results["metadatas"][0]),
    start=1
):
    semantic_rows.append({
        "rank": rank,
        "document": doc_text,
        "distance": distance,
        "category": metadata.get("category"),
        "language": metadata.get("language"),
    })

df_semantic = pd.DataFrame(semantic_rows)
df_semantic

## 12) Comparación: búsqueda exacta vs búsqueda semántica

Esta es una de las celdas más importantes del notebook.

Queremos que se vea clarísimo que:
- la búsqueda exacta depende del texto literal,
- la búsqueda semántica recupera documentos cercanos en significado.


In [ ]:
comparison = pd.DataFrame({
    "query": [query_keyword, query_semantic],
    "search_type": ["keyword", "semantic"],
    "top_result": [
        keyword_results.iloc[0]["text"],
        df_semantic.iloc[0]["document"]
    ]
})

comparison

## 13) Calcular similitud coseno manualmente

ChromaDB ya te devuelve distancias, pero como ingenieras e ingenieros de LLM también conviene entender que, por debajo, hablamos de cercanía matemática entre vectores.

Aquí calcularemos similitud coseno de forma explícita.


In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

query_vec = np.array(query_embedding)

rows = []
for doc, emb in zip(documents, embeddings):
    sim = cosine_similarity(query_vec, np.array(emb))
    rows.append({
        "id": doc["id"],
        "text": doc["text"],
        "cosine_similarity": sim
    })

df_similarity = pd.DataFrame(rows).sort_values(by="cosine_similarity", ascending=False).reset_index(drop=True)
df_similarity

## 14) Filtrado por metadata

Aquí damos un pequeño adelanto de arquitectura profesional.

No siempre quieres buscar en todos los documentos.
A veces primero filtras por:
- categoría,
- idioma,
- cliente,
- región,
- fecha,
- tipo de documento.

Vamos a filtrar por categoría `soporte`.


In [ ]:
query_support = "¿Me pueden devolver mi dinero?"
query_support_embedding = get_embeddings([query_support])[0]

support_results = collection.query(
    query_embeddings=[query_support_embedding],
    n_results=3,
    where={"category": "soporte"},
    include=["documents", "distances", "metadatas"]
)

support_rows = []
for rank, (doc_text, distance, metadata) in enumerate(
    zip(support_results["documents"][0], support_results["distances"][0], support_results["metadatas"][0]),
    start=1
):
    support_rows.append({
        "rank": rank,
        "document": doc_text,
        "distance": distance,
        "category": metadata.get("category"),
        "language": metadata.get("language"),
    })

pd.DataFrame(support_rows)

## 15) Mini caso de negocio

Imagina un sistema de soporte:

El usuario pregunta:
> **"¿Me devuelven mi dinero?"**

Tal vez no existe ningún documento con esa frase exacta.  
Pero sí puede existir uno como:

- **"Política de reembolso para clientes"**
- **"Condiciones de devolución de productos"**

Una vector database ayuda a recuperar esa intención semántica.  
Y eso es justamente una de las bases de RAG.


## 16) Conclusiones finales

### Idea central
Una vector database:

- no se limita a buscar texto exacto,
- almacena embeddings,
- compara cercanía matemática,
- y recupera significado.

### En una frase
**Traditional search finds words.  
Vector search finds meaning.**

### Puente al siguiente tema
Este es uno de los bloques fundamentales para entender cómo funciona **RAG**:
1. representas significado con embeddings,
2. lo guardas en una vector database,
3. recuperas lo más relevante,
4. y se lo pasas al modelo para responder.


## 17) Retos sugeridos

Si quieres extender este notebook:

1. Agrega más documentos con distintas categorías  
2. Prueba queries en español e inglés  
3. Cambia el `n_results` para ver más o menos resultados  
4. Agrega una visualización 3D  
5. Conecta este notebook con uno de RAG completo
